## Module 01: Spark Architecture & Cluster Topology

---

### 1. High-Level Overview
Apache Spark is a **distributed computing system** based on a **Master-Slave Architecture**. It allows you to process massive datasets by splitting the work across a cluster of computers.

### 2. The Core Components

#### A. The Driver (The "Master" Process)
* **Definition:** The central controller of a Spark Application.
* **Key Tasks:**
    * Runs the `main()` function and creates the `SparkSession`.
    * **DAG Creation:** Converts Python code into a logical "Direct Acyclic Graph" (The Plan).
    * **Task Scheduling:** Breaks the DAG into stages and tasks, then distributes them to Executors.
    * **Monitoring:** Tracks job progress and hosts the Spark UI.

#### B. The Cluster Manager (The "Resource Negotiator")
* **Definition:** An external service that manages the physical resources of the cluster.
* **Common Managers:** YARN (Hadoop), Kubernetes, or Spark Standalone.
* **Role:** Allocates CPU and RAM to the Spark application upon request from the Driver.

#### C. The Executor (The "Worker" Process)
* **Definition:** A distributed process that runs on Worker Nodes.
* **Key Tasks:**
    * **Execution:** Runs the specific tasks assigned by the Driver.
    * **Storage:** Manages "Storage Memory" for **Caching and Persistence**.
    * **Reporting:** Sends success/failure status and heartbeats back to the Driver.

---

### 3. Critical Concepts for the Architect

#### I. Directed Acyclic Graph (DAG)
The DAG is the logical sequence of operations Spark plans to perform. It is "Directed" (has a start and end) and "Acyclic" (no loops). It allows Spark to optimize the execution path before any data is actually touched.



#### II. Fault Tolerance (Lineage)
Spark tracks the history of how a dataset was built (Lineage). If an **Executor** fails and data is lost, the **Driver** uses the DAG/Lineage to re-compute that specific missing piece of data on a different Executor.
* **Benefit:** No need to restart the entire 10-hour job if one machine fails.

#### III. Caching vs. Persistence
* **Caching:** Storing a DataFrame in the **Executor's RAM**.
* **Persistence:** Storing data in RAM, Disk, or both.
* **Why?** It prevents the "Expensive Read" problem. Instead of reading from a slow hard drive (S3/HDFS) multiple times, Spark reads it once and keeps it in memory for instant access in later steps.



---

### 4. Summary Table

| Component | Responsibility | Analogous To |
| :--- | :--- | :--- |
| **Driver** | Planning & Coordinating | The Architect |
| **Cluster Manager** | Resource Allocation | The Site Manager |
| **Executor** | Processing & Storing | The Construction Worker |
| **Worker Node** | Physical Hardware | The Construction Site |

---

### 5. Self-Assessment (Review Questions)

1. **Who generates the DAG?** * *Answer:* The Driver.
2. **Where does Cached data live?** * *Answer:* In the Executor's Storage Memory.
3. **What happens if an Executor dies?** * *Answer:* The Driver detects it and re-runs the lost tasks on another Executor using Lineage.
4. **Is the Driver involved in actual data processing?** * *Answer:* No, it only coordinates. If it starts processing data, it creates a bottleneck (and likely crashes).

## Module 02: Lazy Evaluation & Execution Flow

---

### 1. Concept: Lazy Evaluation
In many programming languages, code is "Eager"—it runs the moment it is called. In PySpark, execution is **Lazy**. 

**Definition:** Spark does not execute the instructions immediately. Instead, it builds up a lineage of operations (the DAG) and waits until an **Action** is called before it actually processes any data.

### 2. The Two Types of Operations
To master Spark, you must distinguish between these two categories:

| Feature | Transformations | Actions |
| :--- | :--- | :--- |
| **Definition** | Operations that create a new DataFrame from an existing one. | Operations that trigger computation and return a result. |
| **Behavior** | **Lazy:** They only update the DAG. | **Eager:** They trigger the execution of the DAG. |
| **Output** | Returns a new DataFrame. | Returns a value (int, list) or writes to disk. |
| **Examples** | `filter()`, `select()`, `join()`, `groupBy()`, `map()` | `show()`, `count()`, `collect()`, `save()`, `first()` |



---

### 3. Why Lazy Evaluation? (The Benefits)

#### I. Query Optimization (The Catalyst Optimizer)
Since Spark knows the entire "plan" before it starts, it can optimize it. 
* **Predicate Pushdown:** If you filter data at the end of your script, Spark moves that filter to the very beginning (at the data source level) so it reads less data from the disk.
* **Column Pruning:** If you only use 2 columns out of 100, Spark will only read those 2 columns from the file.

#### II. Fault Tolerance
Because Spark has a recorded "recipe" (the Lineage) of how to create the data, if a machine fails, it simply looks at the recipe and re-runs only the missing parts.



---

### 4. Practical Example (The "PNR" Scenario)

Imagine you are working on your `pnr_etl_project`:

```python
# 1. Transformation (Lazy - No work done)
raw_df = spark.read.parquet("pnr_data.parquet")

# 2. Transformation (Lazy - No work done)
filtered_df = raw_df.filter(raw_df.status == "CONFIRMED")

# 3. Action (Eager - NOW Spark starts the cluster!)
print(filtered_df.count())

## Module 03: Narrow vs. Wide Dependencies (The Shuffle)

---

### 1. The Core Concept: Data Movement
In a distributed system, the "Cost" of an operation is determined by whether data stays on the same machine or has to travel across the network. This relationship between parent and child DataFrames is called a **Dependency**.

### 2. Narrow Dependencies (Low Cost)
A Narrow Dependency exists when each partition of the parent DataFrame is used by **at most one** partition of the child DataFrame.

* **Behavior:** No data movement across the network. All work happens within the Executor's local memory.
* **Analogy:** "Parallel Play." Each worker stays in their own lane and finishes their task without talking to others.
* **Operations:** `filter()`, `map()`, `union()`, `select()`, `drop()`.



### 3. Wide Dependencies (High Cost / The Shuffle)
A Wide Dependency exists when data from multiple parent partitions is needed to build a single child partition. This triggers a **Shuffle**.

* **The Shuffle:** This is the process of redistributing data across the cluster so that data with the same key (e.g., the same `pnr_id`) ends up on the same Executor.
* **Why it's slow:**
    1. **Disk I/O:** Data is often written to local disk before being moved.
    2. **Network I/O:** Massive amounts of data travel over the network cables.
    3. **Serialization:** Converting objects into bytes to send them over the wire.
* **Operations:** `groupBy()`, `join()`, `distinct()`, `repartition()`, `orderBy()`.



---

### 4. Comparison Summary

| Feature | Narrow Dependency | Wide Dependency (Shuffle) |
| :--- | :--- | :--- |
| **Data Movement** | None (Local to Executor) | High (Network Transfer) |
| **Performance** | Extremely Fast | Slower / Resource Intensive |
| **Network Impact** | Zero | High |
| **Failure Recovery** | Fast (Only re-run local task) | Slower (May need to re-shuffle) |

---

### 5. Best Practice: The "Filter First" Rule
To optimize your PNR project, always remember:
**Filter your data (Narrow) before you Join or Group it (Wide).**
* *Example:* If you only need "Confirmed" PNRs, filter them out first. This way, when the "Shuffle" happens, you are moving 100MB

## Module 04: The Execution Hierarchy (Jobs, Stages, and Tasks)

---

### 1. The Anatomy of Execution
When you run a PySpark script, Spark does not execute the code line-by-line in a simple fashion. It breaks your request into a three-level hierarchy to manage distributed work efficiently.

#### Level 1: The Job (The "What")
* **Trigger:** Every time an **Action** (e.g., `.collect()`, `.show()`, `.save()`) is called in your code, Spark generates a Job.
* **Scope:** A Job represents the entire set of computations needed to return a result to the Driver or write it to storage.

#### Level 2: The Stage (The "Boundary")
* **Definition:** A Job is divided into Stages.
* **The Rule:** A new Stage is created whenever a **Shuffle (Wide Dependency)** occurs. 
* **Pipeline:** As long as operations are Narrow (filter, map), Spark stays in the same Stage (this is called "Pipeline Optimization"). As soon as data needs to move across the network (join, groupBy), a "Stage Boundary" is created.

#### Level 3: The Task (The "Unit of Work")
* **Definition:** The smallest unit of execution. 
* **Execution:** A Task is a single operation performed on a **single partition** of data.
* **Parallelism:** If you have 100 partitions, Spark will launch 100 Tasks. These tasks are sent to the Executors to run in parallel.



---

### 2. A Visual Workflow Example

Consider this logic for your PNR project:
`df.filter(...).select(...).groupBy("status").count()`

1. **The Job:** Created by the `.count()` action.
2. **Stage 1:** Includes `read`, `filter`, and `select`. These are all Narrow, so they happen together on the same machine.
3. **The Shuffle:** Spark redistributes data so all "Confirmed" PNRs move to the same Executor.
4. **Stage 2:** Performs the final aggregation on the shuffled data.
5. **Tasks:** If your input file has 10 partitions, Stage 1 will have 10 Tasks.



---

### 3. Summary for Debugging (Spark UI)

| Component | What to look for in the Spark UI |
| :--- | :--- |
| **Job** | Check if you have more Jobs than expected (usually means too many Actions). |
| **Stage** | Too many Stages indicate excessive Shuffling (Wide Dependencies). |
| **Task** | If some Tasks take 10 minutes and others 1 second, you have **Data Skew**. |

---

### 4. Key Terminology

* **Pipelining:** Collapsing multiple transformations into a single Stage to avoid moving data.
* **Shuffle Map Stage:** A stage that writes shuffle files to disk for the next stage to read.
* **Result Stage:** The final stage that sends data back to the Driver.

---

### 5. Self-Assessment (Review Questions)

1. **How many Jobs are created by this code: `df.filter(..).show(); df.groupBy(..).count().show()`?**
   * *Answer:* 2 Jobs (one for each `.show()`).
2. **What determines the number of Tasks in a Stage?**
   * *Answer:* The number of **Partitions** in the DataFrame for that stage.
3. **If a job has no Wide Dependencies (only filters and selects), how many Stages will it have?**
   * *Answer:* 1 Stage.
4. **Why is it bad to have 1 million small Tasks?**
   * *Answer:* Because each task has "scheduling overhead." The time spent by the Driver to manage the task might be longer than the task itself!

## Module 05: Data Partitioning (Managing Parallelism)

---

### 1. Concept: What is a Partition?
A partition is a logical division of your data. In Spark, data is not processed as one giant block; it is split into chunks so multiple CPUs can work on it at once.
* **Key Rule:** 1 Partition = 1 Task = 1 CPU Slot.
* **Goal:** You want enough partitions to keep all your CPUs busy, but not so many that Spark spends all its time managing tasks.

### 2. In-Memory Partitioning: Repartition vs. Coalesce
There are two ways to change the number of partitions of a DataFrame in memory.

| Feature | `repartition(n)` | `coalesce(n)` |
| :--- | :--- | :--- |
| **Primary Purpose** | Increase or decrease partitions. | Decrease partitions only. |
| **Data Movement** | **Full Shuffle:** Data is moved across the entire network. | **Minimal Movement:** Merges existing partitions locally. |
| **Performance** | Expensive (Slow). | Efficient (Fast). |
| **Data Balance** | Perfectly balanced/even. | May result in uneven partition sizes. |
| **Best Use Case** | When you need to increase parallelism or fix "Data Skew." | Right before saving a file to reduce the number of output files. |



---

### 3. Partitioning on Disk (`partitionBy`)
When writing data to a Data Lake (S3, HDFS, or Local Folder), you can organize the physical files based on a column.

* **Method:** `df.write.partitionBy("city").parquet("output_path")`
* **Result:** It creates a folder structure: `output_path/city=London/`, `output_path/city=Delhi/`, etc.
* **Benefit: Partition Pruning.** If a future query filters for `city == 'London'`, Spark will skip all other folders and only read the files inside the London folder.

---

### 4. Summary & Best Practices
1. **The "Sweet Spot":** Aim for partitions to be between **128MB and 256MB** in size.
2. **Avoid Tiny Files:** If you have 1000 tiny partitions but only 10MB of data, use `.coalesce(1)` before writing to disk.
3. **Data Skew:** If one executor is taking 10x longer than others, use `.repartition()` on a high-cardinality column to balance the load.
4. **Hardware Rule:** A common starting point is **2 to 4 partitions per CPU core** in your cluster.



---

### 5. Self-Assessment (Review Questions)

1. **Which command is better for reducing 1000 partitions down to 100?**
   * *Answer:* `.coalesce(100)`, because it avoids a full network shuffle.
2. **Can you use `coalesce()` to increase the number of partitions?**
   * *Answer:* No. If you try to `coalesce` to a higher number, Spark will simply keep the current number of partitions. You must use `repartition()`.
3. **What is the main benefit of `partitionBy` when saving data?**
   * *Answer:* It enables **Partition Pruning**, which significantly speeds up future read operations by ignoring irrelevant data folders.
4. **Why is `repartition()` expensive?**
   * *Answer:* Because it triggers a **Full Shuffle**, meaning every row of data may potentially be moved across the network to a new Executor.

---

# Module 06: Shared Variables (Broadcast & Accumulators)

---

## 1. The Core Problem: Network Inefficiency

In a distributed cluster, when you use a standard variable in a function (like a `map` or `filter`), Spark ships a copy of that variable to **every single task**.

- **Scaling Issue:**  
  If you have 1,000 tasks and a 10MB lookup table, Spark moves **10GB of data** across your network.

- **Solution:**  
  Shared variables allow you to control how data is cached or updated across the executors to minimize network traffic and track global state.

---

## 2. Broadcast Variables (Read-Only Cache)

A **Broadcast Variable** allows you to keep a read-only copy of a dataset cached on **each executor machine** rather than shipping a copy with every task.

### A. The Broadcast Join (Performance King)

In a standard join, Spark shuffles both datasets.  
If one dataset is small (e.g., City Codes) and the other is giant (e.g., Millions of PNRs), we use a **Broadcast Join**.  

- Spark sends the small table to every executor **once**, keeping it in RAM.  
- The large table stays put, and the join happens locally.

### Implementation Example

```python
# 1. Define your small lookup data (Dimension Table)
city_map = {"DL": "Delhi", "NY": "New York", "LDN": "London"}

# 2. Broadcast it (Sent once per Executor)
broadcast_cities = spark.sparkContext.broadcast(city_map)

# 3. Access inside a transformation using .value
def get_city_name(code):
    # This lookup happens in the Executor's local RAM, not the network
    return broadcast_cities.value.get(code, "Unknown")

# Example usage with an RDD or DataFrame
# rdd.map(lambda x: get_city_name(x.city_code))


# Module 07: PySpark SQL & DataFrame API

## 1. What is the DataFrame API?

A **DataFrame** is a distributed collection of data organized into named columns.  
Conceptually, it is equivalent to:
- A table in a relational database  
- A DataFrame in R/Pandas  

But with much richer optimizations under the hood.

### 🔹 RDD vs. DataFrame
- **RDDs**: A collection of "Java/Python objects" where Spark doesn’t know what’s inside.  
- **DataFrames**: Have a **Schema** (column names and types), allowing Spark to understand the data structure.  

### 🔹 Uniformity
Whether you use Python, Scala, or Java, the performance is identical because the code is compiled into the same underlying execution plan.

---

## 2. The Catalyst Optimizer (The Secret Sauce)

The primary reason DataFrames are faster than RDDs is the **Catalyst Optimizer**.  
When you write DataFrame code, Spark does not execute it immediately. Instead, it passes through several stages of optimization:

- **Analysis**: Verifies column names and table existence against the *Catalog*.  
- **Logical Optimization**: Applies rules like:
  - Predicate Pushdown (filtering data as early as possible)  
  - Column Pruning (reading only the required columns)  
- **Physical Planning**: Generates multiple strategies for execution (e.g., Shuffle Join vs. Broadcast Join) and picks the one with the lowest *cost*.  

---

## 3. Creating DataFrames

You can create DataFrames from various sources like **CSV, JSON, Parquet, or existing RDDs**.

```python
# Reading a PNR CSV file with schema inference
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("pnr_data.csv")

# Defining a manual schema (Best practice for production)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("pnr_id", StringType(), True),
    StructField("passenger_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("status", StringType(), True)
])

df_custom = spark.read.schema(schema).csv("pnr_data.csv")


# **Module 08: Data Cleaning & Handling Missing Values**

## 1️⃣ The Challenge: Why Data Cleaning?

In big data pipelines, **"Garbage In, Garbage Out"** is a reality.  
PNR (Passenger Name Record) data often contains:

- **Missing Values (Nulls):** Passengers who didn't provide an email  
- **Duplicates:** The same booking sent twice due to network retries  
- **Inconsistent Formats:** Dates written as `DD-MM-YYYY` *and* `MM/DD/YYYY` within the same file 

## 2️⃣ Handling Missing Data (Nulls)
In PySpark, we use the `df.na` sub-module to manage null values.

### A. Dropping Nulls — `dropna()`

In [ ]:
# Drop any row that contains at least one null value
df_clean = df.dropna()

# Drop rows only if 'pnr_id' or 'email' is null
df_clean = df.dropna(subset=["pnr_id", "email"])

# Drop rows only if ALL columns are null
df_clean = df.dropna(how="all")

### B. Filling Nulls — `fillna()`

In [ ]:
# Fill all null strings with "Unknown" and null numbers with 0
df_filled = df.fillna("Unknown").fillna(0)

# Fill specific columns with specific values
df_filled = df.fillna({"age": 25, "city": "Not Disclosed"})

## 3️⃣ Removing Duplicates

In [ ]:
# Remove absolute duplicate rows (identical in every column)
df_unique = df.dropDuplicates()

# Remove duplicates based on a specific key (e.g., PNR ID)
# This keeps the first occurrence it finds
df_unique_pnr = df.dropDuplicates(["pnr_id"])

## 4️⃣ Advanced Cleaning: The Imputer Strategy

In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=["age", "fare"], 
    outputCols=["age_imputed", "fare_imputed"]
).setStrategy("mean")

df_imputed = imputer.fit(df).transform(df)

## 5️⃣ Column Transformations with `F.when()`

In [ ]:
from pyspark.sql import functions as F

# Flagging invalid ages
df_validated = df.withColumn("age_status", 
    F.when(F.col("age") < 0, "Invalid")
     .when(F.col("age") > 120, "Out of Range")
     .otherwise("Valid")
)

## 6️⃣ Summary Table: Cleaning Methods

| Method | Purpose | Best Used When... |
|--------|---------|------------------|
| `dropna()` | Removes rows | The missing data makes the row useless |
| `fillna()` | Replaces nulls | You need a default value to prevent errors |
| `dropDuplicates()` | Removes repeats | Data ingestion produced redundant records |
| `Imputer` | Statistical fill | You want to preserve data distribution (Mean/Median) |


## 7️⃣ Self-Assessment (MCQ)

**Q1:** What is the risk of using `df.dropna()` without a subset?  
✔ **Answer:** You might lose a significant amount of data because a row is deleted if even a single, non-essential column is null.

---

**Q2:** How is `dropDuplicates(["id"])` different from `dropDuplicates()`?  
✔ **Answer:** `dropDuplicates()` looks for an exact match across all columns; adding a subset like `["id"]` removes rows where *only that* ID matches.

---

**Q3:** Which function is used for IF-ELSE logic in PySpark DataFrames?  
✔ **Answer:** `F.when().otherwise()`


# **Module 09: Spark Built-in Functions (`pyspark.sql.functions`)**

## 1️⃣ Introduction to Spark Functions

PySpark provides **hundreds of built-in functions** that run efficiently at the JVM level, avoiding Python overhead.

We import them using:

```python
from pyspark.sql import functions as F
```

Using `F` is a **universal convention** for readability and efficiency.


## 2️⃣ String Functions

Used for cleaning names, IDs, and descriptions in PNR data.

In [ ]:
# Trimming, Lowercasing, and Substrings
df_strings = df.select(
    F.lower(F.col("passenger_name")).alias("name_lower"),
    F.trim(F.col("pnr_id")).alias("pnr_trimmed"),
    F.substring(F.col("pnr_id"), 0, 3).alias("pnr_prefix")
)

# Concatenation (Joining columns)
df_concat = df.withColumn("full_route", 
    F.concat_ws(" -> ", F.col("origin"), F.col("destination"))
)

## 3️⃣ Date & Timestamp Functions

Spark treats timestamps as **first-class objects**, making date math easier.

In [ ]:
# Formatting and Date Math
df_dates = df.withColumn("booking_date", F.to_date(F.col("raw_date"), "dd-MM-yyyy"))              .withColumn("travel_month", F.month(F.col("booking_date")))              .withColumn("days_until_travel", F.datediff(F.col("travel_date"), F.col("booking_date")))              .withColumn("next_week", F.date_add(F.col("booking_date"), 7))

## 4️⃣ Mathematical & Statistical Functions

Used for financial calculations and data profiling.

In [ ]:
# Rounding and Absolute Values
df_math = df.select(
    F.round(F.col("fare"), 2).alias("rounded_fare"),
    F.ceil(F.col("fare")).alias("next_dollar"),
    F.sqrt(F.col("variance")).alias("std_dev")
)

# Aggregation-based functions (used with groupBy)
df.groupBy("origin").agg(
    F.avg("fare").alias("avg_fare"),
    F.max("fare").alias("max_fare"),
    F.stddev("fare").alias("fare_spread")
)

## 5️⃣ Array & Complex Type Functions

Useful when handling structured or nested JSON data.

In [ ]:
# Handling list of tags or luggage items
# e.g. ['Suitcase', 'Backpack']
df_arrays = df.withColumn("first_item", F.col("luggage_list")[0])               .withColumn("item_count", F.size(F.col("luggage_list")))               .withColumn("has_suitcase", F.array_contains(F.col("luggage_list"), "Suitcase"))

## 6️⃣ Why Use `F` Instead of Python `math` or `datetime`?

| Reason | Explanation |
|--------|------------|
| **Serialization** | Python functions force data to move between Python ↔ JVM, slowing execution |
| **Optimization** | `pyspark.sql.functions` run directly inside Spark execution plans |
| **Scalability** | Works on distributed data without collecting to Python memory |


## 7️⃣ Summary Table: Function Categories

| Category | Common Functions | Use Case |
|----------|-----------------|---------|
| String | `trim`, `lower`, `regexp_replace` | Cleaning text & regex matching |
| Date | `to_date`, `datediff`, `add_months` | Flight scheduling & time windows |
| Math | `round`, `floor`, `abs` | Fare calculations & discounts |
| Collection | `explode`, `array_contains`, `size` | Nested JSON or lists |


## 8️⃣ Self-Assessment (MCQ)

**Q1:** What is the benefit of using `F.concat_ws` over `+` for strings?  
✔ **Answer:** `concat_ws` handles null values safely; `+` makes the string null if any value is null.

---

**Q2:** How do you convert `"2025-12-30"` into a Spark date type?  
✔ **Answer:** `F.to_date(col_name, "yyyy-MM-dd")`

---

**Q3:** Which function counts items inside an array column?  
✔ **Answer:** `F.size()`


# **Module 10: User Defined Functions (UDFs)**

## 1️⃣ What is a UDF?

A **User Defined Function (UDF)** lets you write **custom Python logic** and apply it inside a Spark DataFrame.  
Even though Spark provides many built‑in functions, there are times when you need domain-specific logic —  
for example:
- Custom PNR decryption
- A special tax rule
- Airline-dependent fare bucket classification

## 2️⃣ How UDFs Work — *The Python Barrier*

Unlike built-in functions that run **inside the JVM**, Python UDFs require Spark to:

1. Serialize the data from the JVM  
2. Send it to a **separate Python process**  
3. Execute the Python code  
4. Serialize results back to JVM

This creates overhead — making UDFs slower than built-in functions.

## 3️⃣ Implementing a UDF

### A. Using the Decorator (Recommended)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Define custom logic
@udf(returnType=StringType())
def categorize_age(age):
    if age is None: return "Unknown"
    if age < 18: return "Minor"
    if age < 60: return "Adult"
    return "Senior"

# Apply to DataFrame
df_classified = df.withColumn("age_group", categorize_age(df["age"]))

### B. Registering for SQL Use

In [ ]:
# Register UDF for SQL usage
spark.udf.register("sql_categorize", categorize_age)

# Using inside Spark SQL
spark.sql("SELECT pnr_id, sql_categorize(age) FROM pnr_table").show()

## 4️⃣ ⚠️ The *Performance Killer* Warning

Standard UDFs are the **slowest Spark transformation** because:

| Issue | Description |
|--------|------------|
| ❌ No Optimization | Spark's Catalyst optimizer **can't look inside** a UDF |
| 🐌 Python Barrier | Data moves JVM → Python → JVM for every row |
| 📉 Black Box | Prevents predicate pushdown & execution plan tuning |

> **Rule of Thumb:** Use a UDF **only if** no alternative built‑in function exists.


## 5️⃣ Faster Option: Pandas UDFs (Vectorized UDFs)

If you must use Python logic, Pandas UDFs are faster because they operate on **batches of data** using **Apache Arrow**.

In [ ]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def vectorized_discount(price: pd.Series) -> pd.Series:
    return price * 0.9  # Vectorized operation

df_discounted = df.withColumn("new_price", vectorized_discount(df["price"]))

## 6️⃣ Comparison Summary

| Feature | Built‑in Functions | Pandas UDF (Vectorized) | Standard Python UDF |
|---------|-------------------|--------------------------|---------------------|
| Performance | 🚀 Fastest | ⚡ Fast | 🐢 Slow |
| Optimization | ✔ Full | ➖ Partial | ❌ None |
| Execution | Native JVM | Apache Arrow batches | Row-by-row Python |
| Complexity | Easy | Medium | Easy |


## 7️⃣ Self‑Assessment (MCQ)

**Q1:** Why are standard Python UDFs inefficient?  
✔ **Answer:** Because they serialize data row‑by‑row between Python & JVM, blocking optimizations.

---

**Q2:** What speeds up Pandas UDFs?  
✔ **Answer:** **Apache Arrow** — transfers data in batches.

---

**Q3:** When should you prefer built‑in functions over UDFs?  
✔ **Answer:** **Always** — UDFs should be used only as a last resort.


# **Module 11: Joins & Performance Tuning**

## 1️⃣ How Joins Work in Distributed Systems

In single-machine databases, joins are straightforward.  
But in **Spark**, data is distributed across executors, so Spark must ensure that rows sharing the same join key are located on the **same executor**.

> 📌 **This process is called _Co‑location_.**

Only after co‑location can Spark perform the join operation efficiently.

## 2️⃣ Shuffle Hash Join (Standard Join) — *Default Behavior*

This is the join strategy used when **both DataFrames are large**.

### 🔄 How it works
1. **Shuffle Phase** — Spark hashes the join key and moves data across the cluster so rows with the same key end up on the same partition  
2. **Sort / Hash Phase** — Data is sorted or hashed locally so Spark can join matching rows

### ⚠️ Problem
**Shuffling is expensive** — it requires:
- Disk I/O
- Data serialization/deserialization
- Network transfer latency

Therefore, shuffle joins are **slower and costlier**.

## 3️⃣ Broadcast Hash Join — *When One Side is Small*

If one table is **small (<100MB)**, Spark can broadcast it to all executors and join it locally.  
This eliminates shuffling of the large table.

### ✨ Benefits
| Feature | Broadcast Join |
|---------|----------------|
| Move small data to all nodes | ✔ Yes |
| Shuffle large table | ❌ No |
| Network cost | 🔽 Very Low |
| Performance | 🚀 Fastest join strategy |

In [ ]:
from pyspark.sql.functions import broadcast

# Broadcast-based join
joined_df = large_pnr_df.join(broadcast(small_codes_df), "status_id")

## 4️⃣ Handling Data Skew

**Data Skew** happens when one join key (e.g., `"Unknown"` or `"NYC"`) appears far more frequently than others.  
This can overload one executor while others are idle.

### 🛠️ Solutions
| Technique | Description | Best Use Case |
|-----------|-------------|---------------|
| **Salting** | Append random number to key to distribute load | Single dominant key |
| **Filtering** | Drop null or high-frequency keys before join | When skew caused by invalid data |

## 5️⃣ Join Types in PySpark

Spark supports standard SQL join types.

In [ ]:
# Inner Join (Default) - Only matching keys
df.join(df2, "id", "inner")

# Left Outer - All rows from left, matching from right
df.join(df2, "id", "left")

# Full Outer - All rows from both sides
df.join(df2, "id", "outer")

# Left Anti - Rows in left with NO match in right
df.join(df2, "id", "left_anti")

## 6️⃣ Comparison: Join Performance Characteristics

| Join Strategy | Data Size Requirement | Network Traffic | Performance |
|---------------|-----------------------|----------------|------------|
| **Broadcast Join** | One side must be small | 🔽 Low | 🚀 Fastest |
| **Shuffle Hash Join** | Both large | 🔼 High | 🐌 Slow |
| **Sort-Merge Join** | Default for large tables | 🔼 High | ⚖ Scalable |

## 7️⃣ Self‑Assessment (MCQ)

**Q1:** What is the primary benefit of a Broadcast Join?  
✔ **Answer:** It eliminates the shuffle phase and avoids moving large data over the network.

---

**Q2:** What happens if you try to broadcast a 10GB DataFrame?  
✔ **Answer:** You may get an Out of Memory (OOM) error — each executor must store a full copy.

---

**Q3:** Which join type finds rows in Table A that do *not* exist in Table B?  
✔ **Answer:** `Left Anti Join`

# **Module 12: Optimization — Caching & Persisting**

## 1️⃣ The Problem: Recomputation

In Spark, every time you call an **Action** (like `count()`, `show()`, `save()`), Spark **recalculates the entire DAG** from the source.

### 🎯 Scenario
You read **1TB of data**, clean it, and then:
- Count rows
- Find average fare
- Save results

### ❌ Without Optimization
Spark processes **1TB three times**, wasting:
- CPU cycles  
- Disk I/O  
- Network bandwidth  

> **Caching solves this problem by storing results after first computation.**


## 2️⃣ Caching vs Persisting

### A. `cache()` — Store in Memory Only

In [ ]:
# Cache a filtered DataFrame
df_cleaned = df.filter(df["age"] > 18).cache()

# Trigger computation & load into memory
df_cleaned.count()  # first Action => caching happens here

### B. `persist()` — Flexible Storage Levels

In [ ]:
from pyspark import StorageLevel

# Store in RAM first, spill to Disk if RAM is full
df_optimized = df.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger persist
df_optimized.count()

## 3️⃣ When Should You Cache or Persist?

Caching **everything** harms performance due to memory pressure.

### ✅ Cache only when:
- You reuse a DataFrame multiple times
- It results from **expensive transformations** (joins, UDFs)
- You run **iterative ML workloads**

> 🧠 *Caching saves time when recomputation is heavier than memory cost.*

## 4️⃣ Freeing Memory — `unpersist()`

In [ ]:
# Manually release memory
df_optimized.unpersist()

## 5️⃣ Comparison: `cache()` vs `persist()`

| Feature | `cache()` | `persist()` |
|---------|----------|-------------|
| Default Level | `MEMORY_ONLY` | `MEMORY_AND_DISK` |
| Custom Storage Choice | ❌ No | ✔ Yes |
| Replication | ❌ No | ✔ Yes (`MEMORY_ONLY_2`) |


## 6️⃣ Important Optimization Tip: Action Trigger

Caching is **lazy** — nothing is stored until you call an **Action**.

```python
df.cache()     # Does NOT store anything yet
df.count()     # Now Spark computes & caches
```

> ⚡ Always follow caching with an action to materialize it.


## 7️⃣ Self‑Assessment (MCQ)

**Q1:** What happens if you `cache()` a DataFrame but never call an Action?  
✔ **Answer:** Nothing. Spark never stores it.

---

**Q2:** Which storage level is safest for datasets that exceed RAM?  
✔ **Answer:** `MEMORY_AND_DISK`

---

**Q3:** How do you manually remove cached data?  
✔ **Answer:** `.unpersist()`
